## Load EmoBank Dataset

In [ ]:
import requests
import pandas as pd
from pathlib import Path

In [ ]:
# Download EmoBank dataset if not already present
def download_emobank():
    """Download the EmoBank dataset from the official source."""
    url = "https://github.com/JULIELab/EmoBank/raw/master/corpus/emobank.csv"
    
    # Save to current working directory
    file_path = Path("emobank.csv")
    
    if not file_path.exists():
        print("Downloading EmoBank dataset...")
        response = requests.get(url)
        response.raise_for_status()
        
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(response.text)
        print(f"Downloaded EmoBank dataset to {file_path}")
    else:
        print(f"EmoBank dataset already exists at {file_path}")
    
    return file_path

# Download the dataset
emobank_path = download_emobank()

# Load the EmoBank dataset
df = pd.read_csv(emobank_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nFirst few rows:")
display(df.head())

# Check for missing values
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
import itertools
from scipy import stats

# Define a grid of bin sizes for each dimension
v_bin_options = [5, 10, 20, 30, 40, 50, 100, 250, 500, 750, 1000]
a_bin_options = [5, 10, 20, 30, 40, 50, 100, 250, 500, 750, 1000]
d_bin_options = [5, 10, 20, 30, 40, 50, 100, 250, 500, 750, 1000]
sample_size = 50

results = []

for v_bins, a_bins, d_bins in itertools.product(v_bin_options, a_bin_options, d_bin_options):
    df_test = df.copy()
    df_test['V_bin'] = pd.cut(df_test['V'], bins=v_bins, labels=False)
    df_test['A_bin'] = pd.cut(df_test['A'], bins=a_bins, labels=False)
    df_test['D_bin'] = pd.cut(df_test['D'], bins=d_bins, labels=False)
    df_test['strata'] = (
        df_test['V_bin'].astype(str) + '_' +
        df_test['A_bin'].astype(str) + '_' +
        df_test['D_bin'].astype(str)
    )
    strata_counts = df_test['strata'].value_counts()
    strata_proportions = strata_counts / len(df_test)
    sampled_dfs = []
    for stratum in strata_counts.index:
        stratum_data = df_test[df_test['strata'] == stratum]
        n_samples = max(1, int(strata_proportions[stratum] * sample_size))
        if len(stratum_data) >= n_samples:
            sampled_strata = stratum_data.sample(n=n_samples, random_state=42)
        else:
            sampled_strata = stratum_data
        sampled_dfs.append(sampled_strata)
    test_sample = pd.concat(sampled_dfs, ignore_index=True)
    if len(test_sample) > sample_size:
        test_sample = test_sample.sample(n=sample_size, random_state=42)
    # Calculate similarity metrics
    ks_pvalues = {}
    for dim in ['V', 'A', 'D']:
        ks_stat, ks_pvalue = stats.ks_2samp(df[dim], test_sample[dim])
        ks_pvalues[dim] = ks_pvalue
    results.append({
        'v_bins': v_bins,
        'a_bins': a_bins,
        'd_bins': d_bins,
        'ks_pvalue_V': ks_pvalues['V'],
        'ks_pvalue_A': ks_pvalues['A'],
        'ks_pvalue_D': ks_pvalues['D'],
    })

# Create DataFrame of results
results_df = pd.DataFrame(results)

# Filter for combinations where all p-values > 0.05
passing = results_df[
    (results_df['ks_pvalue_V'] > 0.05) &
    (results_df['ks_pvalue_A'] > 0.05) &
    (results_df['ks_pvalue_D'] > 0.05)
]

print("Bin size combinations with p-value > 0.05 for all dimensions:")
display(passing)

In [ ]:
passing.to_csv('Data/bin_combinations.csv', index=False)

In [ ]:
passing = pd.read_csv('Data/bin_combinations.csv')

In [ ]:
# Slice the first row of every combination of V, A, and D p-values that are equal
first_rows = passing.drop_duplicates(subset=['ks_pvalue_V', 'ks_pvalue_A', 'ks_pvalue_D']).copy()

# Identify the minimnum p-value among the three and add it as a column
first_rows['min_pvalue'] = first_rows[['ks_pvalue_V', 'ks_pvalue_A', 'ks_pvalue_D']].min(axis=1)
first_rows.sort_values(by='min_pvalue', ascending=False, inplace=True)
first_rows

In [ ]:
# Find the row with the largest min_pvalue
best_config = first_rows.loc[first_rows['min_pvalue']==first_rows['min_pvalue'].max()].copy()
best_config.reset_index(drop=True, inplace=True)

print(f"Best configuration:")
print(f"V bins: {best_config['v_bins'][0]}")
print(f"A bins: {best_config['a_bins'][0]}")
print(f"D bins: {best_config['d_bins'][0]}")

In [ ]:
best_config

In [ ]:
# Find the row with the largest min_pvalue
best_config = first_rows.loc[first_rows['min_pvalue']==first_rows['min_pvalue'].max()].copy()
best_config.reset_index(drop=True, inplace=True)

print(f"Best configuration:")
print(f"V bins: {best_config['v_bins'][0]}")
print(f"A bins: {best_config['a_bins'][0]}")
print(f"D bins: {best_config['d_bins'][0]}")

# Generate sample using the best configuration
df_best = df.copy()
df_best['V_bin'] = pd.cut(df_best['V'], bins=best_config['v_bins'][0], labels=False)
df_best['A_bin'] = pd.cut(df_best['A'], bins=best_config['a_bins'][0], labels=False)
df_best['D_bin'] = pd.cut(df_best['D'], bins=best_config['d_bins'][0], labels=False)
df_best['strata'] = (
    df_best['V_bin'].astype(str) + '_' +
    df_best['A_bin'].astype(str) + '_' +
    df_best['D_bin'].astype(str)
)

# Sample from each stratum
strata_counts = df_best['strata'].value_counts()
strata_proportions = strata_counts / len(df_best)
sampled_dfs = []

for stratum in strata_counts.index:
    stratum_data = df_best[df_best['strata'] == stratum]
    n_samples = max(1, int(strata_proportions[stratum] * sample_size))
    
    if len(stratum_data) >= n_samples:
        sampled_strata = stratum_data.sample(n=n_samples, random_state=42)
    else:
        sampled_strata = stratum_data
    
    sampled_dfs.append(sampled_strata)

# Combine and limit to sample_size
best_sample = pd.concat(sampled_dfs, ignore_index=True)
if len(best_sample) > sample_size:
    best_sample = best_sample.sample(n=sample_size, random_state=42)

# Remove helper columns and keep only original columns
best_sample_clean = best_sample[['id', 'split', 'V', 'A', 'D', 'text']].reset_index(drop=True)

In [ ]:
print(f"\nBest sample size: {len(best_sample_clean)}")
print(f"\nBest sample V, A, D statistics:")
print(best_sample_clean[['V', 'A', 'D']].describe())
print(df[['V', 'A', 'D']].describe())

In [ ]:
best_sample_clean.to_csv('Data/annotation_sample.csv', index=False)